# Part 25: Scikit-learn Core API

**Dataset:** Hotel Booking Demand (Antonio et al., 2019) -- 117,429 real bookings from two Portuguese hotels.

**What you will learn:**
- The two core sklearn abstractions: Transformer and Predictor
- Feature scaling with StandardScaler and MinMaxScaler
- Train/test splitting without data leakage
- Fitting LinearRegression and LogisticRegression
- Evaluating models with cross-validation and learning curves
- Saving a trained model with joblib

**Prerequisites:** Parts 22-24 (ML Landscape, ML Taxonomy, ML Workflow).

## 1. Setup

All imports at the top -- PEP 8 import ordering enforced by `ruff --select I`.
`from __future__ import annotations` enables the modern union syntax `X | Y` on Python 3.9.

In [ ]:
from __future__ import annotations

# stdlib
from pathlib import Path
from typing import Any

# tables
from great_tables import GT, md as gt_md

# persistence
import joblib

# visualisation
from lets_plot import (
    LetsPlot,
    aes,
    as_discrete,
    geom_bar,
    geom_hline,
    geom_line,
    geom_point,
    geom_ribbon,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
)

# logging
from loguru import logger

# data
import numpy as np
import pandas as pd

# sklearn -- models
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

# sklearn -- metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# sklearn -- model selection
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline

# sklearn -- transformers
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# project branding
from ark.plot.gt_style import metrics_report, themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, BRAND_PALETTE, DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html()

### 1.1 Project constants

Paths and URLs as SCREAMING_SNAKE_CASE module-level constants -- easy to find and change without searching function bodies. Covered in Part 12 (Dev Tools: code style).

In [ ]:
DATA_URL: str = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
DATA_PATH: Path = Path("data/hotel_bookings.csv")
MODEL_DIR: Path = Path("models")

### 1.2 Define the data loading function

A typed function wraps the download-and-cache logic. The function is idempotent: it downloads once; subsequent calls load from disk.

In [ ]:
def load_hotel_data(
    path: Path = DATA_PATH,
    url: str = DATA_URL,
) -> pd.DataFrame:
    """Load hotel bookings, downloading from url on first call."""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = pd.read_csv(url)
        raw.to_csv(path, index=False)
        logger.success(f"Downloaded {len(raw):,} rows to {path}")
    else:
        raw = pd.read_csv(path)
        logger.info(f"Loaded {len(raw):,} rows from cache: {path}")
    df = raw[(raw["adr"] > 0) & (raw["adr"] <= 1000)].reset_index(drop=True)
    logger.info(f"After cleaning: {len(df):,} rows")
    return df

### 1.3 Load the data

Execute the loader. `df` now holds 117 k cleaned hotel booking records.

In [ ]:
df: pd.DataFrame = load_hotel_data()

### 1.4 Inspect shape and class balance

In [ ]:
logger.info(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
logger.info(f"Hotels in dataset: {df['hotel'].value_counts().to_dict()}")
logger.info(f"Overall cancellation rate: {df['is_canceled'].mean():.1%}")

### 1.5 Summary statistics

A branded GT table gives a cleaner overview than `df.describe()`.

In [ ]:
stats: pd.DataFrame = df.describe().round(2).T.reset_index().rename(columns={"index": "feature"})
(
    themed_gt(GT(stats), n_rows=len(stats))
    .tab_header(
        title=gt_md("**Hotel Bookings -- Summary Statistics**"),
        subtitle="117 k bookings from two Portuguese hotels (2015-2017)",
    )
    .tab_source_note("Antonio et al., 2019 | Data in Brief")
)

## 2. The sklearn API Contract

Before we touch the hotel data, we need to understand the two abstractions that every sklearn algorithm inherits from.

<div style="font-family:monospace;background:#F8FAFC;border:1px solid #E2E8F0;border-radius:8px;padding:16px 20px;margin:12px 0;font-size:13px;line-height:2">
<b>Transformer</b> (e.g. StandardScaler)<br>
&nbsp;&nbsp;<code>scaler.fit(X_train)</code> &rarr; <i>learn &mu; and &sigma; from training data only</i><br>
&nbsp;&nbsp;<code>scaler.transform(X)</code> &rarr; <i>apply those parameters to any split</i><br><br>
<b>Predictor</b> (e.g. LinearRegression)<br>
&nbsp;&nbsp;<code>model.fit(X_train, y_train)</code> &rarr; <i>learn weights from labeled data</i><br>
&nbsp;&nbsp;<code>model.predict(X)</code> &rarr; <i>return predicted values or labels</i>
</div>

**From Part 24 (ML Workflow, Section 2):** the distinction between "learning parameters" and "applying parameters" maps directly onto the train-vs-test discipline covered in the workflow chapter.

### 2.1 Transformer -- the fit step

We use a toy array to isolate the API from the hotel data.

In [ ]:
demo_x: np.ndarray = np.array([[1.0, 100.0], [2.0, 200.0], [3.0, 300.0]])
demo_scaler: StandardScaler = StandardScaler()
demo_scaler.fit(demo_x)
logger.info(f"Learned mean:  {demo_scaler.mean_}")
logger.info(f"Learned scale: {demo_scaler.scale_}")

### 2.2 Transformer -- the transform step

Apply the learned parameters to a new point. The scaler uses the same mean and scale every time -- only the input changes.

In [ ]:
new_point: np.ndarray = np.array([[4.0, 400.0]])
scaled_point: np.ndarray = demo_scaler.transform(new_point)
logger.info(f"Raw input:  {new_point}")
logger.info(f"Scaled:     {scaled_point}")

### 2.3 Predictor -- fit with labels, then predict

A Predictor learns weights from labeled data `(X, y)` and maps new `X` values to predicted `y` values.

In [ ]:
demo_y: np.ndarray = np.array([1.0, 2.0, 3.0])
demo_lr: LinearRegression = LinearRegression()
demo_lr.fit(demo_x, demo_y)
demo_pred: np.ndarray = demo_lr.predict(new_point)
logger.info(f"Predicted y: {demo_pred}")

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; Transformer vs Predictor

A **Transformer** has `fit` + `transform`. It never sees `y`. A **Predictor** has `fit(X, y)` + `predict(X)`. A **Pipeline** chains Transformers + one final Predictor, fitting all stages with a single `pipeline.fit(X_train, y_train)` call.
:::

## 3. The Hotel Booking Dataset

**From Part 23 (What Is Machine Learning?, Section 2):** supervised learning requires a clearly defined prediction target. This dataset gives us two natural targets.

| Target | Column | ML Task |
|--------|--------|---------|
| Room price | `adr` | Regression |
| Booking cancelled | `is_canceled` | Binary Classification |

### 3.1 Define prediction targets

In [ ]:
TARGET_REG: str = "adr"  # Average Daily Rate in euros (continuous)
TARGET_CLF: str = "is_canceled"  # 0 = kept, 1 = cancelled (binary)

### 3.2 Select numerical features

We start with six interpretable columns. Part 26 extends this to a full mixed feature set.

In [ ]:
FEATURES: list[str] = [
    "lead_time",  # days booked in advance
    "stays_in_weekend_nights",  # weekend nights stayed
    "stays_in_week_nights",  # weekday nights stayed
    "adults",  # number of adults
    "previous_cancellations",  # prior cancellations by this guest
    "booking_changes",  # post-booking modifications
]

### 3.3 Build feature matrix X and target vectors

`to_numpy()` produces a plain NumPy array. We specify `dtype=float` to catch implicit integer promotion early.

In [ ]:
X: np.ndarray = df[FEATURES].to_numpy(dtype=float)
y_reg: np.ndarray = df[TARGET_REG].to_numpy(dtype=float)
y_clf: np.ndarray = df[TARGET_CLF].to_numpy(dtype=int)
logger.info(f"X: {X.shape}  y_reg: {y_reg.shape}  y_clf: {y_clf.shape}")

### 3.4 Explore the regression target: ADR distribution

**From Part 23 (Section 3.1):** check the distribution of the target before modelling. Skewed targets may benefit from a log transform.

In [ ]:
adr_df: pd.DataFrame = df[["adr"]].copy()
(
    ggplot(adr_df, aes(x="adr"))
    + geom_bar(stat="bin", fill=INFO, alpha=0.85, bins=60)
    + labs(
        title="Distribution of Average Daily Rate (ADR)",
        subtitle="Regression target: room price per night in euros",
        x="ADR (euros)",
        y="Count",
    )
    + modern_theme(grid=True)
)

### 3.5 Explore the classification target: cancellation by hotel type

**From Part 23 (Section 4):** class imbalance affects which metrics are meaningful. The chart reveals whether both hotels share the same base rate.

In [ ]:
cancel_df: pd.DataFrame = (
    df.groupby("hotel", as_index=False)["is_canceled"]
    .agg(total="count", canceled="sum")
    .assign(rate=lambda d: d["canceled"] / d["total"])
)
(
    ggplot(cancel_df, aes(x="hotel", y="rate", fill="hotel"))
    + geom_bar(stat="identity", alpha=0.85)
    + scale_fill_manual(values=[INFO, WARNING])
    + labs(
        title="Cancellation Rate by Hotel Type",
        subtitle="37% overall -- City Hotel cancels more than Resort",
        x="Hotel Type",
        y="Cancellation Rate",
    )
    + modern_theme(legend_pos="none")
)

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 1: lead time vs cancellation

Compute the median lead time for cancelled and non-cancelled bookings.

```python
# Your code here
```

**Hint:** `df.groupby('is_canceled')['lead_time'].median()`
:::

## 4. Feature Scaling

**From Part 24 (ML Workflow, Section 4):** features on very different scales cause gradient-based algorithms to update weights unevenly, slowing convergence or finding a suboptimal solution.

<div style="font-family:monospace;background:#FEF2F2;border:1px solid #FECACA;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
<b style="color:#DC2626">Wrong (leaky):</b><br>
&nbsp;&nbsp;scaler.fit_transform(X_all) then split &rarr; scaler saw test rows<br><br>
<b style="color:#059669">Correct:</b><br>
&nbsp;&nbsp;split first, then scaler.fit(X_train) and scaler.transform(X_test)
</div>

The leakage diagram above is the key rule: **fit the scaler on training data only**, then apply to both sets.

### 4.1 Feature ranges before scaling

Look at the raw min/max for each feature to see the scale mismatch.

In [ ]:
for i, feat in enumerate(FEATURES):
    logger.info(f"  {feat:<30} min={X[:, i].min():>7.1f}  max={X[:, i].max():>7.1f}")

### 4.2 Split the data before fitting any scaler

**Splitting before scaling** is the most important ordering constraint in the workflow.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y_reg, test_size=0.2, random_state=42)
logger.info(f"Train: {X_tr.shape}  Test: {X_te.shape}")

### 4.3 Fit StandardScaler on training data only

`fit` learns mean and std from `X_tr`. These values describe the training distribution -- the test set should not influence them.

In [ ]:
scaler: StandardScaler = StandardScaler()
scaler.fit(X_tr)
logger.info(f"Learned means:  {scaler.mean_.round(2)}")
logger.info(f"Learned stds:   {scaler.scale_.round(2)}")

### 4.4 Apply the fitted scaler to train and test

Same scaler object, same parameters, different input arrays.

In [ ]:
X_train_s: np.ndarray = scaler.transform(X_tr)
X_test_s: np.ndarray = scaler.transform(X_te)
for i, feat in enumerate(FEATURES):
    logger.info(f"  {feat:<30} min={X_train_s[:, i].min():>6.2f}  max={X_train_s[:, i].max():>6.2f}")

### 4.5 Alternative: MinMaxScaler

MinMaxScaler maps every feature to [0, 1]. Prefer it for algorithms that need bounded inputs (neural networks, some distance-based methods).

In [ ]:
mms: MinMaxScaler = MinMaxScaler()
mms.fit(X_tr)
X_mm: np.ndarray = mms.transform(X_tr)
for i, feat in enumerate(FEATURES):
    logger.info(f"  {feat:<30} min={X_mm[:, i].min():.2f}  max={X_mm[:, i].max():.2f}")

::: {.callout-warning icon=false}
## <i class="bi bi-bug-fill" style="color:#DC2626"></i>&nbsp; Common Mistake: fitting the scaler before splitting

```python
# WRONG: scaler sees test rows
X_scaled = StandardScaler().fit_transform(X)
X_train_s, X_test_s = X_scaled[:n_train], X_scaled[n_train:]

# CORRECT: split first
X_tr, X_te, y_tr, y_te = train_test_split(X, y)
sc = StandardScaler().fit(X_tr)  # only training data
X_train_s = sc.transform(X_tr)
X_test_s  = sc.transform(X_te)
```
:::

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 2: RobustScaler

Fit a `RobustScaler` (uses median and IQR instead of mean and std -- more robust to outliers).

```python
from sklearn.preprocessing import RobustScaler
# Your code here
```

**Question:** which feature changes the most compared to StandardScaler?
:::

## 5. Establishing Baselines

**From Part 24 (ML Workflow, Section 5):** always start with a trivial baseline. If a learned model cannot beat the baseline, something is wrong with the features, the split, or the training procedure.

sklearn's `DummyRegressor` and `DummyClassifier` implement the dumbest possible strategies.

### 5.1 Regression baseline: always predict the training mean

In [ ]:
dummy_reg: DummyRegressor = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train_s, y_tr)
dummy_pred: np.ndarray = dummy_reg.predict(X_test_s)
dummy_mae: float = mean_absolute_error(y_te, dummy_pred)
dummy_const: float = float(dummy_reg.constant_.flat[0])
logger.info(f"Dummy always predicts: {dummy_const:.2f} euros")
logger.info(f"Dummy MAE: {dummy_mae:.2f}")

### 5.2 Classification split and baseline

Stratify on the binary target to preserve the class ratio in both splits.

In [ ]:
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
sc_clf: StandardScaler = StandardScaler()
Xc_train_s: np.ndarray = sc_clf.fit_transform(Xc_tr)
Xc_test_s: np.ndarray = sc_clf.transform(Xc_te)
logger.info(f"Train cancel rate: {yc_tr.mean():.2%}  Test cancel rate: {yc_te.mean():.2%}")

### 5.3 Classification baseline: always predict the majority class

In [ ]:
dummy_clf: DummyClassifier = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(Xc_train_s, yc_tr)
dummy_clf_pred: np.ndarray = dummy_clf.predict(Xc_test_s)
dummy_f1: float = f1_score(yc_te, dummy_clf_pred, zero_division=0)
majority_class: int = int(dummy_clf.classes_[np.argmax(dummy_clf.class_prior_)])
logger.info(f"Dummy always predicts class: {majority_class}")
logger.info(f"Dummy F1: {dummy_f1:.3f}")

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; Why baselines matter

A model that achieves 63% accuracy on a dataset where 63% belong to one class has learned **nothing** -- the baseline achieves the same score. Accuracy is misleading for imbalanced problems; F1 score (or precision-recall AUC) exposes the baseline for what it is.
:::

## 6. Linear Models

**From Part 23 (What Is Machine Learning?, Section 1.3):** linear models sit at the "high interpretability, low capacity" end of the model spectrum. Their coefficients are directly interpretable as feature importances.

We use `LinearRegression` for ADR (continuous) and `LogisticRegression` for cancellation (binary).

### 6.1 Define a typed evaluation helper

One function covers all regression comparisons throughout the notebook.

In [ ]:
def evaluate_regressor(
    model: Any,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    name: str,
) -> dict[str, Any]:
    """Return MAE, RMSE, and R2 for a fitted regression model."""
    preds: np.ndarray = model.predict(X_eval)
    return {
        "Model": name,
        "MAE": float(mean_absolute_error(y_eval, preds)),
        "RMSE": float(mean_squared_error(y_eval, preds) ** 0.5),
        "R2": float(r2_score(y_eval, preds)),
    }

### 6.2 Fit LinearRegression

In [ ]:
lin_reg: LinearRegression = LinearRegression()
lin_reg.fit(X_train_s, y_tr)
logger.success("LinearRegression fitted")

### 6.3 Compare against the baseline

In [ ]:
reg_results: pd.DataFrame = pd.DataFrame(
    [
        evaluate_regressor(dummy_reg, X_test_s, y_te, "DummyRegressor (mean)"),
        evaluate_regressor(lin_reg, X_test_s, y_te, "LinearRegression"),
    ]
)
metrics_report(
    reg_results,
    metrics=["MAE", "RMSE", "R2"],
    minimize_cols=["MAE", "RMSE"],
    maximize_cols=["R2"],
    title="Regression Performance: ADR prediction",
    subtitle="Test set | Hotel Booking Demand dataset",
)

### 6.4 Inspect learned coefficients

Each coefficient: "holding other features constant, how many euros does ADR change when this standardised feature increases by 1?" 

In [ ]:
coef_df: pd.DataFrame = (
    pd.DataFrame({"feature": FEATURES, "coefficient": lin_reg.coef_})
    .sort_values("coefficient", key=abs, ascending=False)
    .reset_index(drop=True)
)
(
    themed_gt(GT(coef_df), n_rows=len(coef_df)).tab_header(
        title=gt_md("**LinearRegression Coefficients**"),
        subtitle="Standardised features -- magnitude = relative importance",
    )
)

### 6.5 Visualise coefficients

In [ ]:
coef_plot: pd.DataFrame = coef_df.assign(
    color=lambda d: d["coefficient"].apply(lambda v: "positive" if v >= 0 else "negative")
)
(
    ggplot(
        coef_plot,
        aes(
            x=as_discrete("feature", order_by="coefficient", order=1),
            y="coefficient",
            fill="color",
        ),
    )
    + geom_bar(stat="identity", alpha=0.85)
    + scale_fill_manual(values={"positive": INFO, "negative": DANGER})
    + labs(
        title="LinearRegression Coefficients",
        subtitle="Standardised features -- positive = higher ADR, negative = lower ADR",
        x="",
        y="Coefficient",
    )
    + modern_theme(x_axis_angle=30, legend_pos="none")
)

### 6.6 Residual plot

Random scatter around zero means the model has no systematic bias. Patterns (curved shape, funnel) indicate the linear model is missing something.

In [ ]:
y_pred_lin: np.ndarray = lin_reg.predict(X_test_s)
resid_df: pd.DataFrame = pd.DataFrame(
    {
        "predicted": y_pred_lin,
        "residual": y_te - y_pred_lin,
    }
)
(
    ggplot(resid_df, aes(x="predicted", y="residual"))
    + geom_point(color=INFO, alpha=0.12, size=1)
    + geom_hline(yintercept=0, color=DANGER, linetype="dashed", size=1)
    + labs(
        title="Residuals vs Predicted ADR",
        subtitle="Random scatter around zero = no systematic bias",
        x="Predicted ADR (euros)",
        y="Residual (actual - predicted)",
    )
    + modern_theme(grid=True)
)

### 6.7 Fit LogisticRegression for cancellation

`class_weight="balanced"` adjusts the loss inversely proportional to class frequencies. Without it, the model learns to predict "not cancelled" for almost everything because that minimises raw error on an imbalanced dataset.

**From Part 23 (Section 4):** class imbalance is one of the most common failure modes in binary classification.

In [ ]:
log_reg: LogisticRegression = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)
log_reg.fit(Xc_train_s, yc_tr)
logger.success("LogisticRegression fitted")

### 6.8 Evaluate the classifier with classification_report

In [ ]:
yc_pred: np.ndarray = log_reg.predict(Xc_test_s)
report: str = classification_report(yc_te, yc_pred, target_names=["Kept", "Cancelled"])
logger.info("\n" + report)

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 3: Ridge vs LinearRegression

Ridge adds an L2 penalty that shrinks coefficients toward zero, reducing overfitting.

```python
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_tr)
# compare evaluate_regressor(ridge, ...) with lin_reg
```

**Question:** does Ridge lower the gap between train and test MAE?
:::

## 7. Cross-Validation

**From Part 24 (ML Workflow, Section 5):** a single train/test split gives a noisy, variance-prone estimate of generalisation. Cross-validation rotates the held-out subset across the entire training set.

<div style="font-family:monospace;background:#F8FAFC;border:1px solid #E2E8F0;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
Fold 1: [<b>val</b>][train][train][train][train] &rarr; score<sub>1</sub><br>
Fold 2: [train][<b>val</b>][train][train][train] &rarr; score<sub>2</sub><br>
Fold 3: [train][train][<b>val</b>][train][train] &rarr; score<sub>3</sub><br>
Fold 4: [train][train][train][<b>val</b>][train] &rarr; score<sub>4</sub><br>
Fold 5: [train][train][train][train][<b>val</b>] &rarr; score<sub>5</sub><br>
Final estimate = mean(score<sub>1..5</sub>) +/- std
</div>

Each fold acts as a validation set exactly once. The final estimate is `mean +/- std` across all folds.

### 7.1 Define the regression pipeline for CV

We wrap scaler + model in a `Pipeline` so the scaler is re-fit on each fold's training subset automatically -- no leakage.

In [ ]:
reg_pipe: Pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

### 7.2 Run KFold cross-validation for regression

In [ ]:
kf: KFold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_reg: np.ndarray = cross_val_score(
    reg_pipe,
    X_tr,
    y_tr,
    cv=kf,
    scoring="neg_mean_absolute_error",
)
logger.info(f"CV MAE per fold: {(-cv_reg).round(2)}")
logger.info(f"Mean +/- Std: {-cv_reg.mean():.2f} +/- {cv_reg.std():.2f}")

### 7.3 Define the classification pipeline for CV

In [ ]:
clf_pipe: Pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]
)

### 7.4 Run StratifiedKFold cross-validation for classification

`StratifiedKFold` ensures the class ratio is preserved in every fold.

In [ ]:
skf: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_clf: np.ndarray = cross_val_score(
    clf_pipe,
    Xc_tr,
    yc_tr,
    cv=skf,
    scoring="f1",
)
logger.info(f"CV F1 per fold: {cv_clf.round(3)}")
logger.info(f"Mean +/- Std: {cv_clf.mean():.3f} +/- {cv_clf.std():.3f}")

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; KFold vs StratifiedKFold

**KFold** -- use for regression (continuous targets).

**StratifiedKFold** -- use for classification. Preserves the class ratio in every fold, preventing a fold from being all one class on heavily imbalanced datasets.
:::

## 8. Learning Curves

**From Part 24 (ML Workflow, Section 6 -- Bias-Variance tradeoff):** a learning curve plots train and validation score as training set size grows.

- **Train >> validation, gap shrinks**: high variance (overfitting) -- gather more data or add regularisation.
- **Both scores high error, flat**: high bias (underfitting) -- add features or try a more expressive model.

### 8.1 Compute the learning curve

`learning_curve` re-trains the pipeline at each specified size using CV for the validation score.

In [ ]:
train_sizes: np.ndarray
train_lc_scores: np.ndarray
val_lc_scores: np.ndarray
train_sizes, train_lc_scores, val_lc_scores = learning_curve(
    reg_pipe,
    X_tr,
    y_tr,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
logger.info(f"Training sizes used: {train_sizes}")

### 8.2 Plot the learning curve

In [ ]:
lc_df: pd.DataFrame = pd.DataFrame(
    {
        "n": np.tile(train_sizes, 2),
        "score": np.concatenate([-train_lc_scores.mean(1), -val_lc_scores.mean(1)]),
        "err": np.concatenate([train_lc_scores.std(1), val_lc_scores.std(1)]),
        "split": ["Train"] * len(train_sizes) + ["Validation"] * len(train_sizes),
    }
).assign(
    ymin=lambda d: d["score"] - d["err"],
    ymax=lambda d: d["score"] + d["err"],
)
(
    ggplot(lc_df, aes(x="n", y="score", color="split", fill="split"))
    + geom_line(size=1.2)
    + geom_ribbon(aes(ymin="ymin", ymax="ymax"), alpha=0.15, color=None)
    + scale_color_manual(values={"Train": INFO, "Validation": WARNING})
    + scale_fill_manual(values={"Train": INFO, "Validation": WARNING})
    + labs(
        title="Learning Curve -- LinearRegression on ADR",
        subtitle="Gap between train and validation = variance; flat high error = bias",
        x="Training Set Size",
        y="MAE (lower = better)",
    )
    + modern_theme(grid=True, legend_pos="bottom")
)

::: {.callout-tip icon=false}
## <i class="bi bi-lightbulb-fill" style="color:#8B5CF6"></i>&nbsp; Pro Tip: read the learning curve

If train and validation scores have **converged** and are both high-error: more data will not help -- the model lacks capacity.

If there is a **large gap** that shrinks as n grows: more data reduces variance. The gap tells you roughly how much.
:::

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 4: learning curve for LogisticRegression

Compute and plot a learning curve for `clf_pipe` using `scoring='f1'` and `StratifiedKFold`.

```python
# Your code here
```

**Question:** does the classifier need more data or a more complex model?
:::

## 9. Saving and Loading a Model

**From Part 13 (Dev Tools, Reproducibility):** a trained model is an artefact. Save the full bundle -- scaler, model, and feature list -- so downstream code can reconstruct the prediction pipeline exactly.

### 9.1 Save the training bundle

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
model_path: Path = MODEL_DIR / "lin_reg_adr.joblib"
bundle: dict[str, Any] = {
    "scaler": scaler,
    "model": lin_reg,
    "features": FEATURES,
    "target": TARGET_REG,
}
joblib.dump(bundle, model_path)
logger.success(f"Bundle saved to {model_path}")

### 9.2 Reload and predict on one booking

In [ ]:
loaded: dict[str, Any] = joblib.load(model_path)
sample: np.ndarray = X_te[:1]
sample_scaled: np.ndarray = loaded["scaler"].transform(sample)
predicted: float = float(loaded["model"].predict(sample_scaled)[0])
actual: float = float(y_te[0])
logger.info(f"Predicted ADR: {predicted:.2f}  Actual ADR: {actual:.2f}")

::: {.callout-tip icon=false}
## <i class="bi bi-lightbulb-fill" style="color:#8B5CF6"></i>&nbsp; Pro Tip: save the scaler with the model

Save the scaler **inside** the bundle, never separately. A model evaluated on unscaled inputs in production is one of the most common silent bugs in deployed ML systems. The scaler and model must travel together.
:::

## Capstone: End-to-End Training Function

Combine the full workflow -- split, scale, fit, evaluate, save -- into a single typed function.
This is the design pattern you will use in Part 26 and throughout the MLOps path.

### Define the capstone function

In [ ]:
def train_and_evaluate(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    task: str,
    out_dir: Path = Path("models"),
) -> dict[str, Any]:
    """Run the full train-evaluate-save workflow for one task."""
    X_all: np.ndarray = df[features].to_numpy(dtype=float)
    y_all: np.ndarray = df[target].to_numpy()

    stratify: np.ndarray | None = y_all if task == "classification" else None
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=stratify)
    sc: StandardScaler = StandardScaler()
    Xtr_s: np.ndarray = sc.fit_transform(Xtr)
    Xte_s: np.ndarray = sc.transform(Xte)

    if task == "regression":
        model: Any = LinearRegression()
        baseline: Any = DummyRegressor(strategy="mean")
    else:
        model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
        baseline = DummyClassifier(strategy="most_frequent")

    baseline.fit(Xtr_s, ytr)
    model.fit(Xtr_s, ytr)

    if task == "regression":
        m: dict[str, float] = {
            "baseline_mae": float(mean_absolute_error(yte, baseline.predict(Xte_s))),
            "model_mae": float(mean_absolute_error(yte, model.predict(Xte_s))),
            "model_r2": float(r2_score(yte, model.predict(Xte_s))),
        }
    else:
        m = {
            "baseline_f1": float(f1_score(yte, baseline.predict(Xte_s), zero_division=0)),
            "model_f1": float(f1_score(yte, model.predict(Xte_s))),
        }

    out_dir.mkdir(exist_ok=True)
    save_path: Path = out_dir / f"{target}.joblib"
    joblib.dump({"scaler": sc, "model": model, "features": features, "target": target}, save_path)
    logger.success(f"Saved {task} bundle for '{target}' to {save_path}")
    logger.info(f"Metrics: {m}")
    return {"model": model, "scaler": sc, "metrics": m}

### Run the capstone for regression (ADR)

In [ ]:
reg_result: dict[str, Any] = train_and_evaluate(df, FEATURES, TARGET_REG, task="regression", out_dir=MODEL_DIR)

### Run the capstone for classification (cancellation)

In [ ]:
clf_result: dict[str, Any] = train_and_evaluate(df, FEATURES, TARGET_CLF, task="classification", out_dir=MODEL_DIR)

## Further Reading

- [Scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html) -- comprehensive API reference
- Geron, A. (2022), *Hands-On Machine Learning*, Ch 2-4 -- end-to-end pipeline walkthrough
- Antonio et al. (2019), *Hotel booking demand datasets*, Data in Brief -- dataset reference
- [sklearn Pipeline docs](https://scikit-learn.org/stable/modules/compose.html) -- full ColumnTransformer + Pipeline API (covered in Part 26)

**Next up:** Part 26 (nb05) extends this to a full mixed feature set, introduces `ColumnTransformer` and `SimpleImputer`, and covers GridSearchCV, RandomizedSearchCV, and Optuna hyperparameter optimisation.